# Ling 360 Final Project: Regional Classification of Turkish Folk Songs
**Topic:** Text Classification of Türkü Lyrics using Naive Bayes, N-Gram Models, Subword Tokenization (BPE), and TF-IDF Weighting

### Team Members & Roles
* **Cüneyt Küçük:** Project Manager & Data Analyst - Responsible for data preparation, text normalization, testing and final report generation
* **İbrahim Faruk Ceylan:** Lead Developer, Modeler & Evaluator - Responsible for vectorizer implementation, BPE/TF-IDF pipeline and and evaluation metrics

### Project Objective
The goal of this project is to build a computational model capable of identifying the source region of a Turkish folk song (Türkü) based purely on its textual lyrics.

Following a standard NLP pipeline, we process the raw text and train **Naive Bayes Classifiers** (Multinomial and Complement). Rather than treating this simply as a machine learning task, we systematically test four distinct linguistic hypotheses by altering our feature extraction methods:

1. **Lexical Baseline (Unigrams):** Tests whether isolated lexical items carry sufficient signal to identify the source region.
2. **Syntactic Context (Bigrams):** Tests whether local phrases and syntactic structures provide a stronger regional signature than individual words.
3. **Semantic Filtering (Stopwords):** Tests whether semantically vacuous items (filler words) are irrelevant noise or if they actually serve as crucial geographical markers.
4. **Morphological Analysis (Char N-Grams):** Tests whether regional identity is primarily encoded within the morphology (affixes and suffixes) of this highly agglutinative language.

### Advanced Experimentation
After testing these core hypotheses with the methods learned in class, we implemented two additional tokenization and modeling techniques to see if we could further improve the model's performance:
* **Byte-Pair Encoding (BPE):** A data-driven subword tokenization method used to test if dynamically learning and merging frequent character pairs captures Turkish regional suffixes more efficiently than rigid character n-grams.
* **TF-IDF Weighting:** Term Frequency-Inverse Document Frequency was applied to our morphological model to test if penalizing widespread, standard grammatical sequences while boosting rare, region-specific affixes would yield a higher classification accuracy than raw frequency counts.


## Section 1: Setup and Data Preparation

In [35]:
# =====================================================================
# SECTION 1: SETUP AND DATA PREPARATION
# =====================================================================

# 1. Imports and Environmental Setup
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report, accuracy_score
from tokenizers import ByteLevelBPETokenizer
import nltk
from nltk.corpus import stopwords

# Download required NLTK assets silently to keep the notebook clean
nltk.download('stopwords', quiet=True)


# 2. Data Loading and Consolidation Functions
def load_and_prepare_data(file_path):
    """
    Loads the raw Turkish folk songs dataset, resolves severe class imbalance
    by mapping sparse city labels into 10 macro-regions, and filters
    out unmapped regional targets.

    Parameters:
    -----------
    file_path : str
        The local or remote path to the CSV dataset containing lyrics and regions.

    Returns:
    --------
    pd.DataFrame
        A cleaned and structured DataFrame containing the consolidated 'target_region'
        column, fully optimized for stratified splitting.
    """
    # Load the raw dataset
    df = pd.read_csv(file_path)

    # Target Variable Consolidation (Handling Class Imbalance)
    region_mapping = {
        'Marmara': ['Kırklareli', 'Balıkesir', 'Balıkesi', 'Edirne', 'Çanakkale', 'Sakarya', 'Bilecik', 'Bursa', 'İstanbul', 'Tekirdağ', 'Burs-Bilecik', 'Kocaeli', 'Kırklareli-Trakya'],
        'Ege': ['Kütahya', 'Afyon', 'Manisa', 'Uşak', 'İzmir', 'Aydın', 'Muğla', 'Denizli', 'Burdur-Denizli', 'Ege'],
        'Akdeniz': ['Mersin', 'Antalya', 'Adana', 'Çukurova', 'Burdur', 'Hatay', 'Isparta', 'Kahramanmaraş', 'Osmaniye-Gaziantep'],
        'İç Anadolu': ['Sivas', 'Kırşehir', 'Ankara', 'Yozgat', 'Niğde', 'Kayseri', 'Eskişehir', 'Nevşehir', 'Kırıkkale', 'Orta Anadolu', 'Konya', 'Çankırı', 'Aksaray', 'Kırşehir-Yozgat', 'Orta Anadolu-Orta Toroslar', 'Karaman', 'Sivas-Tokat', 'Tokat-Sivas', 'Malatya-Sivas'],
        'Karadeniz': ['Çorum', 'Tokat', 'Bolu', 'Giresun', 'Trabzon', 'Rize', 'Samsun', 'Ordu', 'Bayburt', 'Kastamonu', 'Amasya', 'Gümüşhane', 'Artvin', 'Karabük', 'Sinop', 'Karadeniz', 'Zonguldak', 'Bartın', 'Karadeniz Ereğlisi', 'Kastamonu-Çankırı', 'Gümüşhane-Bayburt', 'Ordu-Giresun', 'Girsun'],
        'Doğu Anadolu': ['Erzurum', 'Erzincan', 'Malatya', 'Van', 'Elazığ', 'Bitlis', 'Doğu Anadolu', 'Kars', 'Ardahan', 'Ağrı', 'Iğdır', 'Tunceli', 'Muş', 'Hakkari', 'Bingöl', 'Erzurum-Bayburt', 'Erzurum-Kars', 'Kuzeydoğu Anadolu', 'Ardahan-Kars'],
        'Güneydoğu Anadolu': ['Diyarbakır', 'Kilis', 'Güneydoğu Anadolu', 'Şanlıurfa', 'Gaziantep', 'Adıyaman', 'Adıyman', 'Mardin', 'Diyarbakır-Şanlıurfa', 'Siirt', 'Diyarbakır-Erzincan', 'Gaziantep-Adana-Sivas'],
        'Rumeli': ['Rumeli', 'Batı Trakya'],
        'Azerbaycan': ['Azerbaycan', 'Azerbeycan'],
        'Kerkük': ['Kerkük']
    }

    # Reverse the dictionary to map city -> macro_region efficiently
    city_to_macro = {city: macro for macro, cities in region_mapping.items() for city in cities}
    df['target_region'] = df['region'].map(city_to_macro)

    # Drop any songs (like those from Crimea or Cyprus) that do not fit the 10 core classes
    df = df.dropna(subset=['target_region']).copy()

    return df


# 3. Pipeline Execution
# Invoke the modular function to build our primary DataFrame
df = load_and_prepare_data('turkish_folk_songs.csv')

print(f"Total songs ready for processing: {len(df)}")

Total songs ready for processing: 4076


### Text Normalization & Data Splitting (Pre-processing)

* **Case-Folding:** We apply Turkish-safe lowercasing, taking special care with specific characters (`I` and `İ`) to prevent improper mapping to the English `i`.
* **Regular Expressions (Regex):** We utilize the `re` library to strip out punctuation and collapse extra whitespace. This standardizes our vocabulary, ensuring that punctuated words (e.g., "dağlar" and "dağlar.") are interpreted as the exact same token.
* **Stratified Splitting:** We perform an 80/20 train-test split, utilizing stratification to ensure that minority classes (like Kerkük) are proportionally represented in both sets to prevent training bias.

In [36]:
# =====================================================================
# SECTION 2: TEXT NORMALIZATION & DATA SPLITTING
# =====================================================================

def clean_turkish_text(text):
    """
    Normalizes Turkish text by handling case-folding for special characters
    (I/ı, İ/i) and removing punctuation and extra whitespace using RegEx.

    Parameters:
    -----------
    text : str
        The raw lyric text.

    Returns:
    --------
    str
        The cleaned, lowercased, and punctuation-free string.
    """
    if not isinstance(text, str):
        return ""

    # 1. Turkish-safe lowercasing
    text = text.replace('I', 'ı').replace('İ', 'i').lower()

    # 2. RegEx: Substitute any character that is NOT a word character (\w)
    # or whitespace (\s) with a space. This removes punctuation.
    text = re.sub(r'[^\w\s]', ' ', text)

    # 3. RegEx: Collapse multiple sequential spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def prepare_features_and_split(df):
    """
    Applies text normalization to the dataset and performs a stratified
    train/test split to ensure minority classes are proportionally represented.

    Parameters:
    -----------
    df : pd.DataFrame
        The prepared dataframe containing 'song_content' and 'target_region'.

    Returns:
    --------
    tuple
        X_train, X_test, y_train, y_test arrays ready for model training.
    """
    # Apply the normalization function to the lyrics column
    df['clean_lyrics'] = df['song_content'].apply(clean_turkish_text)

    # Define our Features (X) and Target Labels (y)
    X = df['clean_lyrics']
    y = df['target_region']

    # Perform a stratified train/test split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    return X_train, X_test, y_train, y_test

# Execute the pipeline
X_train, X_test, y_train, y_test = prepare_features_and_split(df)

print(f"Training set size: {len(X_train)} | Testing set size: {len(X_test)}")

Training set size: 3260 | Testing set size: 816




## Section 2: Model Training & Experimental Phases

In this section, we train our classifiers. We evaluate two different algorithms:
1. **Multinomial Naive Bayes (MNB):** The standard probabilistic algorithm for text classification.
2. **Complement Naive Bayes (CNB):** An adaptation of MNB specifically designed to correct biases caused by highly imbalanced datasets.

To ensure scientific rigor, our testing is split into two distinct experimental phases.

##### Phase 1: Testing Linguistic Hypotheses
These four models test which structural level of human language carries the strongest regional dialect signal.
* **Model 1 (Lexical Baseline):** `ngram_range=(1,1)`. Assumes regional identity is carried by isolated vocabulary words.
* **Model 2 (Syntactic Context):** `ngram_range=(1,2)`. Assumes regional identity is found in local phrase structures and syntax.
* **Model 3 (Semantic Filter):** Uses custom Turkish stopwords to test whether filler words (*aman, hey, yar*) are irrelevant noise or critical geographical markers.
* **Model 4 (Morphological Analysis):** `analyzer='char_wb'`. Extracts 3-5 character sequences to act as a statistical subword parser, testing if identity is primarily encoded in Turkish affixes.

##### Phase 2: Computational Optimizations
Having established our baseline linguistic features, these two models test advanced computational representations of the text.
* **Model 5 (BPE Subword):** Uses a custom-trained Byte-Pair Encoding tokenizer to test if an AI-driven compression algorithm can capture regional suffixes more efficiently than manual character chunking.
* **Model 6 (TF-IDF Morphological):** Applies Term Frequency-Inverse Document Frequency weighting to our Character N-Grams to test if penalizing standard Turkish grammar while boosting rare regional suffixes yields higher accuracy.

In [37]:
# =====================================================================
# SECTION 3: FEATURE EXTRACTION AND CLASSIFICATION
# =====================================================================

def build_vectorizers(train_data):
    """
    Constructs a dictionary of 6 distinct text vectorizers representing
    different linguistic hypotheses and tokenization strategies.

    Parameters:
    -----------
    train_data : pd.Series
        The training text data required to train the BPE tokenizer vocabulary.

    Returns:
    --------
    dict
        A dictionary mapping descriptive model names to instantiated sklearn
        vectorizer objects.
    """
    # 1. Stopwords Configuration
    turku_stopwords = ['aman', 'yar', 'vay', 'hey', 'de', 'da', 'oy', 'ben', 'sen', 'gülüm', 'bir']
    standard_tr_stopwords = stopwords.words('turkish')
    all_stopwords = list(set(standard_tr_stopwords + turku_stopwords))

    # 2. Train BPE Tokenizer
    print("Training BPE Tokenizer on training data...")
    bpe_tokenizer = ByteLevelBPETokenizer()
    bpe_tokenizer.train_from_iterator(
        train_data.tolist(),
        vocab_size=5000,
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
    )

    # Closure function to pass into the CountVectorizer
    def bpe_tokenize(text):
        return bpe_tokenizer.encode(text).tokens

    # 3. Vectorizer Dictionary Construction
    vecs = {
        "1. Lexical Baseline (Unigrams)": CountVectorizer(min_df=2, ngram_range=(1, 1)),
        "2. Syntactic Context (Bigrams)": CountVectorizer(min_df=2, ngram_range=(1, 2)),
        "3. Semantic Filtering (Stopwords)": CountVectorizer(min_df=2, stop_words=all_stopwords),
        "4. Morphological (Char N-Grams)": CountVectorizer(min_df=2, analyzer='char_wb', ngram_range=(3, 5)),
        "5. Subword (BPE Tokenization)": CountVectorizer(tokenizer=bpe_tokenize, token_pattern=None),
        "6. Morphological (TF-IDF Char N-Grams)": TfidfVectorizer(min_df=2, analyzer='char_wb', ngram_range=(3, 5))
    }

    return vecs

def train_and_evaluate_models(classifier_class, classifier_name, vecs_dict, X_tr, X_te, y_tr, y_te):
    """
    Iterates through all vectorizer configurations, trains a fresh classifier instance,
    and prints a detailed evaluation report.

    Parameters:
    -----------
    classifier_class : class
        The uninstantiated sklearn classifier class (e.g., MultinomialNB).
    classifier_name : str
        The display name of the classifier for the printed report.
    vecs_dict : dict
        The dictionary of prepared vectorizers.
    X_tr, X_te, y_tr, y_te : array-like
        The split training and testing data.
    """
    for model_name, vec in vecs_dict.items():
        print(f"==================================================================")
        print(f"=== {model_name} - {classifier_name} ===")
        print(f"==================================================================")

        # 1. Feature Extraction
        X_train_vec = vec.fit_transform(X_tr)
        X_test_vec = vec.transform(X_te)

        vocab_size = len(vec.get_feature_names_out())
        print(f"Vocabulary/Feature Space Size: {vocab_size}\n")

        # 2. Train the Classifier (Creates a fresh instance each loop)
        model = classifier_class()
        model.fit(X_train_vec, y_tr)

        # 3. Predict on the unseen Test set
        y_pred = model.predict(X_test_vec)

        # 4. Evaluate
        acc = accuracy_score(y_te, y_pred)
        print(f"Exact Match Accuracy: {acc * 100:.2f}%\n")
        print(classification_report(y_te, y_pred, zero_division=0))
        print("\n")


### Phase 1 - Testing Linguistic Hypotheses


In [38]:
# =====================================================================
# PIPELINE EXECUTION - PHASE 1
# =====================================================================

# 1. Build all vectorizers (This will also train the BPE tokenizer in the background)
all_vecs = build_vectorizers(X_train)

# 2. Extract Models 1-4 for the Linguistic Phase
phase_1_linguistic = {k: v for k, v in all_vecs.items() if "1" in k or "2" in k or "3" in k or "4" in k}

print("\n" + "="*70)
print("PHASE 1: TESTING LINGUISTIC HYPOTHESES (Models 1-4)")
print("Goal: Determine which structural level of language carries regional identity.")
print("="*70 + "\n")

# 3. Run BOTH algorithms for Phase 1
train_and_evaluate_models(MultinomialNB, "Multinomial Naive Bayes", phase_1_linguistic, X_train, X_test, y_train, y_test)
train_and_evaluate_models(ComplementNB, "Complement Naive Bayes", phase_1_linguistic, X_train, X_test, y_train, y_test)

Training BPE Tokenizer on training data...

PHASE 1: TESTING LINGUISTIC HYPOTHESES (Models 1-4)
Goal: Determine which structural level of language carries regional identity.

=== 1. Lexical Baseline (Unigrams) - Multinomial Naive Bayes ===
Vocabulary/Feature Space Size: 11510

Exact Match Accuracy: 38.24%

                   precision    recall  f1-score   support

          Akdeniz       0.09      0.05      0.06        64
       Azerbaycan       0.64      0.41      0.50        17
     Doğu Anadolu       0.49      0.55      0.52       176
              Ege       0.35      0.40      0.37        91
Güneydoğu Anadolu       0.32      0.14      0.19        58
        Karadeniz       0.33      0.28      0.30       115
           Kerkük       0.75      0.25      0.38        12
          Marmara       0.28      0.11      0.15        47
           Rumeli       0.60      0.21      0.32        42
       İç Anadolu       0.36      0.58      0.45       194

         accuracy                        

#### **Phase 1 Results**
##### **The Algorithmic Battle: CNB > MNB**

The Majority Class Trap: Multinomial Naive Bayes (MNB) heavily favored regions with the most data (e.g., İç Anadolu), failing to detect smaller classes.

The Fix: Complement Naive Bayes (CNB) successfully corrected this bias, drastically improving minority class recall (e.g., Kerkük, Azerbaycan) by training on the complement of the data.

##### **Linguistic Findings**

Morphology Wins (Model 4): Character N-Grams achieved the highest accuracy (42.52%). Because Turkish is agglutinative, regional dialects manifest in suffixes (gidiyorum vs. gideyrum). Slicing text into 3-5 character chunks successfully captured these markers.

Syntax Adds Noise (Model 2): Moving from Unigrams to Bigrams exploded the feature space but slightly lowered accuracy. Regional identity lives in specific words and suffixes, not syntactic word ordering.

"Stopwords" are Signal (Model 3): Removing filler words (aman, oy, hey) actually hurt performance. In Türkü lyrics, these are critical geographical fingerprints, not noise.

##### **Regional Patterns**

Distinct: Azerbaycan and Kerkük have highly unique vocabularies (high precision).

Blended: Marmara and Akdeniz blend heavily into standard Istanbul Turkish or neighboring dialects (low recall).



### Phase 2: Computational Optimizations

While morphology (Model 4) is the most accurate linguistic feature, it created a massive, computationally expensive space of 45,384 features. Phase 2 tests if we can optimize this:

1. BPE Tokenization (Model 5): Can AI-driven compression learn the true boundaries of regional suffixes more efficiently than rigid 3-5 character chunks?

2. TF-IDF Weighting (Model 6): Can we improve accuracy by statistically penalizing common Turkish grammar (-lar, -yor) while artificially boosting the weight of rare regional suffixes?

In [39]:
# =====================================================================
# PIPELINE EXECUTION - PHASE 2
# =====================================================================

print("\n" + "="*70)
print("PHASE 2: COMPUTATIONAL OPTIMIZATIONS (Models 5-6)")
print("Goal: Test advanced tokenization and statistical weighting on the best linguistic feature.")
print("="*70 + "\n")

phase_2_computational = {k: v for k, v in all_vecs.items() if "5" in k or "6" in k}

# Run BOTH algorithms for Phase 2
train_and_evaluate_models(MultinomialNB, "Multinomial Naive Bayes", phase_2_computational, X_train, X_test, y_train, y_test)
train_and_evaluate_models(ComplementNB, "Complement Naive Bayes", phase_2_computational, X_train, X_test, y_train, y_test)


PHASE 2: COMPUTATIONAL OPTIMIZATIONS (Models 5-6)
Goal: Test advanced tokenization and statistical weighting on the best linguistic feature.

=== 5. Subword (BPE Tokenization) - Multinomial Naive Bayes ===
Vocabulary/Feature Space Size: 4748

Exact Match Accuracy: 39.34%

                   precision    recall  f1-score   support

          Akdeniz       0.21      0.17      0.19        64
       Azerbaycan       0.53      0.59      0.56        17
     Doğu Anadolu       0.50      0.44      0.47       176
              Ege       0.35      0.42      0.38        91
Güneydoğu Anadolu       0.32      0.26      0.29        58
        Karadeniz       0.35      0.33      0.34       115
           Kerkük       0.55      0.50      0.52        12
          Marmara       0.24      0.19      0.21        47
           Rumeli       0.54      0.52      0.53        42
       İç Anadolu       0.41      0.48      0.44       194

         accuracy                           0.39       816
        macro av

#### **Phase 2 Results**
In Phase 2, we tested whether advanced computational representations could improve upon the raw counts of our best linguistic feature (Morphology / Character N-Grams). The results reveal fascinating limitations of standard NLP algorithms when applied to dialectology.

##### **The BPE Trade-off: High Efficiency, Lower Nuance (Model 5)**

The Result: BPE achieved ~39% accuracy across both classifiers.

The Interpretation: While this is lower than our baseline Char N-Gram model (42.52%), it is incredibly impressive when you look at the feature space. BPE compressed the vocabulary from 45,384 features down to just 4,748. It achieved ~92% of the predictive power using only ~10% of the data.

The Catch: BPE is designed for standard compression, not dialectal analysis. It merged frequent characters a bit too aggressively, likely bundling distinct regional suffixes into larger subwords and losing the granular phonetic shifts that the manual 3-5 character chunking caught.

##### **The TF-IDF Collapse: When Math Sabotages Grammar (Model 6)**

The Result: A spectacular failure. Accuracy dropped to 30.15% (MNB) and 35.78% (CNB). The model completely flatlined (0% precision/recall) on minority classes like Akdeniz, Azerbaycan, and Kerkük, while MNB panicked and guessed İç Anadolu for 95% of its predictions.

The Interpretation: The "Inverse Document Frequency" (IDF) penalty ruined the morphological signal. In standard English NLP, IDF is great for penalizing filler words. But in Turkish, the most frequent character sequences (e.g., -lar, -yor, -mek) form the core grammatical backbone of the language. By mathematically penalizing these frequent grammatical chunks, TF-IDF effectively deleted the linguistic foundation the model was relying on to detect regional variations.

## Section 3: Final Project Conclusion

This project successfully developed a computational pipeline to classify the regional origins of Turkish folk songs (Türküler) based purely on their lyrics. By systematically testing both linguistic hypotheses and computational optimizations, we uncovered several critical insights into computational dialectology for agglutinative languages:

**1. Morphology over Syntax**
Regional identity in Turkish folk songs is primarily encoded at the morphological level. The Character N-Gram model (42.52% accuracy) significantly outperformed isolated words and syntactic phrases. This proves that Turkish dialects manifest most strongly in subword suffixes and phonetic shifts rather than distinct root word choices or phrase structures.

**2. Cultural Signals vs. Noise**
Standard NLP practices do not always apply to dialectology. Removing domain-specific "stopwords" (like *aman*, *oy*, *hey*) actually degraded model performance, revealing that these semantically empty filler words are essential geographical fingerprints rather than statistical noise.

**3. Overcoming Algorithmic Bias**
Addressing class imbalance was paramount to the model's success. Multinomial Naive Bayes collapsed into majority-class bias (over-predicting regions like İç Anadolu). However, Complement Naive Bayes successfully neutralized this by training on the inverse of the classes, proving highly effective for imbalanced cultural datasets.

**4. The Limits of Advanced Computation**
More complex math did not guarantee better results for this specific task:
* **TF-IDF Weighting** severely penalized the core grammatical building blocks of Turkish (e.g., frequent suffixes), leading to a catastrophic drop in accuracy.
* **Byte-Pair Encoding (BPE)**, while causing a slight drop in dialectal nuance, proved to be an incredibly powerful compression tool. It shrank the massive character n-gram feature space by nearly 90% while retaining roughly 92% of the predictive power.

**Final Takeaway:** For the computational analysis of highly agglutinative and dialectal text, simpler statistical representations—specifically raw character-level frequencies paired with imbalance-aware probabilistic models like Complement Naive Bayes—provide the most accurate and robust results.

 ## Section 4: Try It Out (Interactive Predictor)

Based on our comprehensive evaluation, **Model 4 (Morphological Char N-Grams) paired with Complement Naive Bayes** achieved the highest accuracy (42.52%) and handled class imbalance the most effectively.

We will now isolate this specific configuration, train a final instance of the model, and use it to predict the regional origin of unseen, custom lyrics.

In [42]:
# 1. Isolate the Best Vectorizer
best_vec_name = "4. Morphological (Char N-Grams)"
best_vec = all_vecs[best_vec_name]

# 2. Train the Final Best Classifier
print(f"Loading best configuration: {best_vec_name} + Complement Naive Bayes...\n")
X_train_best = best_vec.transform(X_train)
best_model = ComplementNB()
best_model.fit(X_train_best, y_train)

# 3. Define the Prediction Function
def predict_turku_region(lyric, trained_model=best_model, trained_vec=best_vec):
    """
    Predicts the regional origin of a Turkish folk song lyric
    using the best-performing Morphological CNB model.
    """
    custom_vec = trained_vec.transform([lyric])
    predicted_region = trained_model.predict(custom_vec)[0]
    probabilities = trained_model.predict_proba(custom_vec)[0]
    class_labels = trained_model.classes_

    print("PREDICTION REPORT")
    print("=" * 60)
    print(f"Input Lyric      : '{lyric}'")
    print(f"Predicted Region : {predicted_region.upper()}")
    print("=" * 60)
    print("\nModel Confidence Across All Regions (Highest to Lowest):")

    confidence_scores = sorted(zip(class_labels, probabilities), key=lambda x: x[1], reverse=True)
    for region, score in confidence_scores:
        marker = "#" if region == predicted_region else "•"
        print(f"   {marker} {region:<20}: {score * 100:.2f}%")

Loading best configuration: 4. Morphological (Char N-Grams) + Complement Naive Bayes...



In [43]:
# 4. Test it out with custom inputs!
predict_turku_region("Seferberlik çıktı seslendi / Mendilim ıslandı mendilim ıslandı / Bu harp senden değil Mevla'dan kaldı / Ağlama validem gider gelürüm / Hakkın helal eyle belkin ölürüm")

PREDICTION REPORT
Input Lyric      : 'Seferberlik çıktı seslendi / Mendilim ıslandı mendilim ıslandı / Bu harp senden değil Mevla'dan kaldı / Ağlama validem gider gelürüm / Hakkın helal eyle belkin ölürüm'
Predicted Region : DOĞU ANADOLU

Model Confidence Across All Regions (Highest to Lowest):
   # Doğu Anadolu        : 72.18%
   • İç Anadolu          : 23.44%
   • Rumeli              : 3.97%
   • Karadeniz           : 0.15%
   • Güneydoğu Anadolu   : 0.09%
   • Kerkük              : 0.05%
   • Azerbaycan          : 0.05%
   • Akdeniz             : 0.04%
   • Marmara             : 0.02%
   • Ege                 : 0.00%
